In [0]:
import datetime as _dt
from pyspark.sql.functions import *
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time


In [0]:
try:
    arrival_date = dbutils.widgets.get("arrival_date")
except Exception:
    arrival_date= _dt.date.today().strftime("%Y-%m-%d")

try:
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = "travel_bookings"

try:
    schema = dbutils.widgets.get("schema")
except Exception:
    schema = "default"

try:
    base_volume = dbutils.widgets.get("base_volume")
except Exception:
    base_volume = f"/Volumes/{catalog}/{schema}/data"

In [0]:
booking_path = f"{base_volume}/booking/bookings_{arrival_date}.csv"
customer_path =  f"{base_volume}/customer/customers_{arrival_date}.csv"

In [0]:
missing = []

try:
    dbutils.fs.ls(booking_path)
except Exception:
    missing.append(booking_path)
try:
    dbutils.fs.ls(customer_path)
except Exception:
    missing.append(customer_path)

if missing:
    raise FileNotFoundError(f"Missing data files: {missing}")

In [0]:
spark.sql(f"create schema if not exists {catalog}.ops")
spark.sql(f"""
          create table if not exists {catalog}.ops.run_log(
              run_id string,
              arrival_date date,
              stage string,
              status string,
              message string,
              recorderd_at timestamp
              ) using delta""")

run_id = f"nb-validate-{arrival_date}-{int(time.time())}"
log_df = spark.createDataFrame([
    (run_id,arrival_date,"validate_inputs","STARTED","Inputs Validated")],["run_id","arrival_date","stage","status","message"])
log_df = log_df.withColumn("arrival_date",F.to_date("arrival_date")).withColumn("recorderd_at",F.current_timestamp())
log_df.write.mode("append").saveAsTable(f"{catalog}.ops.run_log")
print("Validation Succesfull:",booking_path,customer_path)


In [0]:
%sql
select * from travel_bookings.ops.run_log;